# Used Car Price Analysis (India)
Cleaning, merging and visualising four used-car datasets using only **pandas, numpy and matplotlib**.

Run this notebook from the project root (the folder containing `data/` and `graphs/`).

## 1. Imports and settings

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = os.getcwd()  # run the notebook from the project root folder
RAW = os.path.join(BASE, "data", "raw")
CLEAN = os.path.join(BASE, "data", "cleaned")
GRAPHS = os.path.join(BASE, "graphs")
CUR_YEAR = 2026

os.makedirs(CLEAN, exist_ok=True)
os.makedirs(GRAPHS, exist_ok=True)
%matplotlib inline

## 2. Helper functions and loaders
Each file has different column names and units, so each gets its own loader that maps it to one common schema.

In [ ]:
log = []

def note(msg):
    print(msg)
    log.append(msg)


OWNER_MAP = {"first owner": "First", "first": "First", "second owner": "Second", "second": "Second",
             "third owner": "Third", "third": "Third", "fourth & above owner": "Fourth & Above",
             "4 or more": "Fourth & Above", "fourth": "Fourth & Above",
             "test drive car": "Test Drive", "unregistered car": "Unregistered"}
FINAL_COLS = ["brand", "name", "year", "price_inr", "km_driven", "fuel", "seller_type",
              "transmission", "owner", "source"]


def load_dekho():
    d = pd.read_csv(os.path.join(RAW, "CAR_DETAILS_FROM_CAR_DEKHO.csv"))
    d["source"] = "CAR_DETAILS_FROM_CAR_DEKHO"
    return d.rename(columns={"selling_price": "price_inr"})


def load_v3():
    d = pd.read_csv(os.path.join(RAW, "Car_details_v3.csv"))
    d["source"] = "Car_details_v3"
    return d.rename(columns={"selling_price": "price_inr"})


def load_car_data():
    d = pd.read_csv(os.path.join(RAW, "car_data.csv"))
    d = d.rename(columns={"Car_Name": "name", "Year": "year", "Selling_Price": "price_inr",
                          "Kms_Driven": "km_driven", "Fuel_Type": "fuel", "Seller_Type": "seller_type",
                          "Transmission": "transmission", "Owner": "owner"})
    d["price_inr"] = d["price_inr"] * 100000  # lakhs -> INR
    d["owner"] = d["owner"].map({0: "First", 1: "Second", 2: "Third", 3: "Fourth & Above"})
    d["name"] = d["name"].astype(str).str.title()
    d["source"] = "car_data"
    return d


def load_v4():
    d = pd.read_csv(os.path.join(RAW, "car_details_v4.csv"))
    d["name"] = d["Make"].astype(str) + " " + d["Model"].astype(str)
    d = d.rename(columns={"Price": "price_inr", "Year": "year", "Kilometer": "km_driven",
                          "Fuel Type": "fuel", "Seller Type": "seller_type",
                          "Transmission": "transmission", "Owner": "owner"})
    d["source"] = "car_details_v4"
    return d


def standardize(d):
    d = d.copy()
    d["name"] = d["name"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    d["brand"] = d["name"].str.split().str[0].str.title()
    d["brand"] = d["brand"].replace({"Maruti": "Maruti Suzuki"})
    low = d["name"].str.lower()
    d.loc[low.str.startswith("maruti suzuki"), "brand"] = "Maruti Suzuki"
    d.loc[low.str.startswith("land rover"), "brand"] = "Land Rover"
    for c in ("fuel", "seller_type", "transmission"):
        d[c] = d[c].astype(str).str.strip().str.title()
    d["owner"] = d["owner"].astype(str).str.strip().str.lower().map(OWNER_MAP)
    d["seller_type"] = d["seller_type"].replace({"Trustmark Dealer": "Dealer", "Corporate": "Dealer"})
    for c in ("year", "price_inr", "km_driven"):
        d[c] = pd.to_numeric(d[c], errors="coerce")
    return d[FINAL_COLS]

## 3. Load and merge the four datasets

In [ ]:
frames = {"dekho": load_dekho(), "v3": load_v3(), "car_data": load_car_data(), "v4": load_v4()}
for k, f in frames.items():
    note(f"Loaded {k}: {len(f)} rows")

combined = pd.concat([standardize(f) for f in frames.values()], ignore_index=True)
note(f"Combined rows before cleaning: {len(combined)}")

## 4. Data cleaning

In [ ]:
# ---------------- CLEANING ----------------
before = len(combined)
combined = combined.dropna(subset=["name", "year", "price_inr", "km_driven", "fuel", "transmission"])
note(f"Dropped rows with missing key values: {before - len(combined)}")

before = len(combined)
combined = combined[combined["owner"].isin(["First", "Second", "Third", "Fourth & Above"])]
note(f"Dropped test-drive / unregistered / unknown-owner rows: {before - len(combined)}")

before = len(combined)
combined["fuel"] = combined["fuel"].replace({"Cng": "CNG", "Lpg": "LPG"})
combined = combined[combined["fuel"].isin(["Petrol", "Diesel", "CNG", "LPG", "Electric"])]
note(f"Dropped rare/invalid fuel types (hybrid/other): {before - len(combined)}")

before = len(combined)
combined = combined[combined["year"].between(1995, CUR_YEAR)]
combined["year"] = combined["year"].astype(int)
note(f"Dropped impossible years: {before - len(combined)}")

before = len(combined)
combined = combined[(combined["price_inr"] >= 20000) & (combined["km_driven"] > 0)]
note(f"Dropped price < 20,000 or km <= 0: {before - len(combined)}")

before = len(combined)
combined = combined.drop_duplicates(subset=["name", "year", "price_inr", "km_driven", "fuel",
                                            "seller_type", "transmission", "owner"])
note(f"Dropped duplicate listings: {before - len(combined)}")

before = len(combined)
lp = np.log10(combined["price_inr"])
q1, q3 = lp.quantile([.25, .75])
iqr = q3 - q1
mask_p = lp.between(q1 - 2.5 * iqr, q3 + 2.5 * iqr)
k1, k3 = combined["km_driven"].quantile([.25, .75])
mask_k = combined["km_driven"] <= k3 + 4 * (k3 - k1)
combined = combined[mask_p & mask_k]
note(f"Dropped extreme outliers (price / km, IQR rule): {before - len(combined)}")

combined["car_age"] = CUR_YEAR - combined["year"]
combined["price_lakh"] = (combined["price_inr"] / 1e5).round(3)
combined = combined.reset_index(drop=True)
note(f"FINAL cleaned rows: {len(combined)}")
combined.to_csv(os.path.join(CLEAN, "cleaned_cars.csv"), index=False)
with open(os.path.join(BASE, "cleaning_log.txt"), "w") as fh:
    fh.write("\n".join(log) + "\n")

## 5. Visualizations

In [ ]:
# ---------------- PLOTS ----------------
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25})
C = ["#2E86AB", "#E4572E", "#76B041", "#F3A712", "#7B2CBF"]
stats = {}

### Graph 1: Price distribution
Outcome: the market is budget-heavy; median price is about 4.3 lakh.

In [ ]:
# 1. Price distribution
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(combined["price_lakh"], bins=50, color=C[0], edgecolor="white")
med = combined["price_lakh"].median()
ax.axvline(med, color=C[1], ls="--", label=f"Median = {med:.2f} lakh")
ax.set_xlabel("Selling price (INR lakh)")
ax.set_ylabel("Number of cars")
ax.set_title("1. Distribution of Used-Car Selling Prices")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(GRAPHS, "1_price_distribution.png"))
plt.show()
stats["median_price"] = med
stats["pct_under_5"] = (combined["price_lakh"] < 5).mean() * 100

### Graph 2: Average price vs manufacturing year
Outcome: newer cars are worth far more, with the steepest rise after 2019.

In [ ]:
# 2. Average price by manufacturing year
by_year = combined.groupby("year")["price_lakh"].agg(["mean", "count"])
by_year = by_year[by_year["count"] >= 20]
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(by_year.index, by_year["mean"], marker="o", color=C[1], lw=2)
ax.set_xlabel("Manufacturing year")
ax.set_ylabel("Average price (INR lakh)")
ax.set_title("2. Average Price vs Manufacturing Year (years with 20+ listings)")
fig.tight_layout()
fig.savefig(os.path.join(GRAPHS, "2_price_vs_year.png"))
plt.show()
stats["year_min"], stats["year_max"] = by_year.index.min(), by_year.index.max()
stats["price_first_year"], stats["price_last_year"] = by_year["mean"].iloc[0], by_year["mean"].iloc[-1]

### Graph 3: Top 10 brands
Outcome: Maruti Suzuki has the most listings; Toyota is the priciest of the top 10.

In [ ]:
# 3. Top brands: count and avg price
top = combined["brand"].value_counts().head(10).index
bb = combined[combined["brand"].isin(top)].groupby("brand")["price_lakh"].agg(["count", "mean"])
bc = bb.sort_values("count")
bm = bb.sort_values("mean")
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].barh(bc.index, bc["count"], color=C[0])
axes[0].set_title("Listings per brand")
axes[0].set_xlabel("Number of cars")
axes[1].barh(bm.index, bm["mean"], color=C[3])
axes[1].set_title("Average price per brand")
axes[1].set_xlabel("Average price (INR lakh)")
fig.suptitle("3. Top 10 Brands: Popularity vs Price", fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(GRAPHS, "3_top_brands.png"))
plt.show()
stats["top_brand"], stats["top_brand_n"] = bc["count"].idxmax(), int(bc["count"].max())
stats["priciest_brand"], stats["priciest_val"] = bm["mean"].idxmax(), bm["mean"].max()
stats["cheapest_brand"], stats["cheapest_val"] = bm["mean"].idxmin(), bm["mean"].min()

### Graph 4: Price by fuel type and transmission
Outcome: diesel and automatic cars have higher median prices.

In [ ]:
# 4. Fuel & transmission boxplots
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fuels = [f for f in ["Petrol", "Diesel", "CNG"] if (combined["fuel"] == f).sum() >= 20]
axes[0].boxplot([combined.loc[combined["fuel"] == f, "price_lakh"] for f in fuels],
                tick_labels=fuels, showfliers=False, patch_artist=True,
                boxprops=dict(facecolor="#BFD7EA"), medianprops=dict(color=C[1]))
axes[0].set_title("By fuel type")
axes[0].set_ylabel("Price (INR lakh)")
tr = ["Manual", "Automatic"]
axes[1].boxplot([combined.loc[combined["transmission"] == t, "price_lakh"] for t in tr],
                tick_labels=tr, showfliers=False, patch_artist=True,
                boxprops=dict(facecolor="#FCE3B1"), medianprops=dict(color=C[1]))
axes[1].set_title("By transmission")
fig.suptitle("4. Price Spread by Fuel Type and Transmission (outliers hidden)", fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(GRAPHS, "4_fuel_transmission_boxplot.png"))
plt.show()
med_f = combined.groupby("fuel")["price_lakh"].median()
med_t = combined.groupby("transmission")["price_lakh"].median()
stats["med_diesel"], stats["med_petrol"] = med_f.get("Diesel"), med_f.get("Petrol")
stats["med_auto"], stats["med_manual"] = med_t.get("Automatic"), med_t.get("Manual")
stats["pct_manual"] = (combined["transmission"] == "Manual").mean() * 100

### Graph 5: Kilometres driven vs price
Outcome: more kilometres means a lower price, but car age matters more.

In [ ]:
# 5. km driven vs price
fig, ax = plt.subplots(figsize=(8, 5))
samp = combined.sample(min(4000, len(combined)), random_state=1)
ax.scatter(samp["km_driven"] / 1000, samp["price_lakh"], s=8, alpha=.35, color=C[0])
ax.set_yscale("log")
x = combined["km_driven"] / 1000
y = np.log10(combined["price_lakh"])
m, b = np.polyfit(x, y, 1)
xs = np.linspace(x.min(), x.max(), 100)
ax.plot(xs, 10 ** (m * xs + b), color=C[1], lw=2, label="Trend line")
corr = np.corrcoef(combined["km_driven"], combined["price_lakh"])[0, 1]
ax.set_xlabel("Kilometres driven (thousand km)")
ax.set_ylabel("Price (INR lakh, log scale)")
ax.set_title(f"5. Kilometres Driven vs Price (correlation = {corr:.2f})")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(GRAPHS, "5_km_vs_price.png"))
plt.show()
stats["corr_km"] = corr
stats["corr_age"] = np.corrcoef(combined["car_age"], combined["price_lakh"])[0, 1]

## 6. Summary statistics

In [ ]:
print("\nSTATS")
for k, v in stats.items():
    print(k, round(float(v), 3) if isinstance(v, (float, np.floating)) else v)
print(combined["source"].value_counts())